# Module 4.6 — Code Generation & DSLs

### Python Mastery · Track 4: Functional Programming & Metaprogramming

At its most advanced, metaprogramming treats code as data: generating it, analysing it, and transforming it at runtime. This module covers `exec`, `eval`, and `compile`, the abstract syntax tree through the `ast` module, and building small internal domain-specific languages with fluent interfaces. It also stresses the safety considerations these tools demand.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Note the safety warnings carefully; these tools are powerful and can be dangerous.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Use `eval`, `exec`, and `compile`, and explain their differences and risks.
2. Avoid running untrusted input through these functions.
3. Parse and inspect code with the `ast` module.
4. Analyse and transform an abstract syntax tree safely.
5. Design a small internal domain-specific language with a fluent interface.

**Prerequisites:** Tracks 1 to 3, and Modules 4.1 to 4.5.

---

## Part 1 · `eval`, `exec`, and `compile`

Three built-ins run code held in strings. `eval` evaluates a single **expression** and returns its value. `exec` runs one or more **statements** (assignments, loops, definitions) and returns `None`. `compile` turns source text into a code object that `eval` or `exec` can run, which is worthwhile when the same code runs repeatedly.

You can restrict the names available to the code by passing dictionaries for globals and locals.

In [ ]:
# eval handles a single expression and returns its value.
print("eval:", eval("2 + 3 * 4"))

# You can supply the names the expression may use, and nothing else.
print("eval with names:", eval("x * y", {"x": 6, "y": 7}))

# exec runs statements; it does not return a value, but it can change a namespace.
namespace = {}
exec("result = sum(range(5))", namespace)
print("exec produced:", namespace["result"])

# compile builds a reusable code object (here, an expression).
code_obj = compile("a ** 2 + b ** 2", "<generated>", "eval")
print("compiled and evaluated:", eval(code_obj, {"a": 3, "b": 4}))

> **Caution — never run untrusted input.** `eval` and `exec` execute whatever they are given, so passing user input or external data to them can run arbitrary, malicious code (deleting files, leaking data, and worse). Restricting globals helps but is **not** a security boundary. For parsing values safely, use `ast.literal_eval`, shown next. As a rule, avoid `eval`/`exec` unless you fully control the input and have no safer alternative.

In [ ]:
import ast

# ast.literal_eval safely evaluates only literals: numbers, strings, tuples, lists,
# dicts, booleans, and None. It cannot call functions or run arbitrary code.
print(ast.literal_eval("[1, 2, 3]"))
print(ast.literal_eval("{'a': 1, 'b': 2}"))
print(ast.literal_eval("(True, None, 3.14)"))

# A dangerous string is simply rejected rather than executed.
try:
    ast.literal_eval("__import__('os').getcwd()")
except ValueError as e:
    print("safely rejected:", type(e).__name__)

## Part 2 · The Abstract Syntax Tree

Before Python runs your code, it parses the text into an **abstract syntax tree** (AST): a structured representation of the program's grammar. The `ast` module exposes this tree, letting you analyse code without executing it. `ast.parse` builds the tree, and `ast.dump` shows its structure.

In [ ]:
import ast

source = "total = price * quantity + tax"
tree = ast.parse(source)

# A readable dump of the tree's structure (trimmed for clarity).
print(ast.dump(tree, indent=2)[:400], "...")

## Part 3 · Analysing an AST with `NodeVisitor`

Because the AST is just data, you can walk it to learn things about code: which names it uses, which functions it calls, how complex it is. `ast.NodeVisitor` makes this easy: subclass it and define `visit_<NodeType>` methods for the node kinds you care about. This is exactly how linters and static analysers work.

In [ ]:
import ast

class NameCollector(ast.NodeVisitor):
    """Collect every variable name that is read in some code."""
    def __init__(self):
        self.names = set()

    def visit_Name(self, node):
        if isinstance(node.ctx, ast.Load):     # a name being read (not assigned)
            self.names.add(node.id)
        self.generic_visit(node)               # continue into child nodes

source = "result = alpha + beta * gamma - alpha"
tree = ast.parse(source)
collector = NameCollector()
collector.visit(tree)
print("names read:", sorted(collector.names))

In [ ]:
import ast

class CallCounter(ast.NodeVisitor):
    """Count how many function calls appear, and record their names."""
    def __init__(self):
        self.calls = []

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name):
            self.calls.append(node.func.id)
        self.generic_visit(node)

source = "print(len(data)); print(max(data)); save(data)"
counter = CallCounter()
counter.visit(ast.parse(source))
print("calls found:", counter.calls)
print("number of calls:", len(counter.calls))

## Part 4 · Transforming an AST

`ast.NodeTransformer` goes further: it lets you rewrite the tree, returning new nodes in place of old ones. After transforming, `ast.fix_missing_locations` repairs position information, and `compile` turns the modified tree back into runnable code. This is the basis of optimisers and code-rewriting tools. The example below replaces every numeric constant with its doubled value.

In [ ]:
import ast

class DoubleNumbers(ast.NodeTransformer):
    """Replace each numeric literal n with 2 * n."""
    def visit_Constant(self, node):
        if isinstance(node.value, (int, float)):
            return ast.copy_location(ast.Constant(value=node.value * 2), node)
        return node

source = "value = 3 + 4 * 5"
tree = ast.parse(source, mode="exec")
DoubleNumbers().visit(tree)
ast.fix_missing_locations(tree)              # restore line/column info after editing

namespace = {}
exec(compile(tree, "<transformed>", "exec"), namespace)
# Originally 3 + 4*5 = 23; after doubling each number: 6 + 8*10 = 86.
print("transformed result:", namespace["value"])

## Part 5 · Building an Internal DSL with a Fluent Interface

A domain-specific language is a small, focused notation for a particular task. An **internal** DSL is built within Python itself, often as a **fluent interface**: methods that each return `self`, so calls chain into a readable, sentence-like form. This needs no `eval` or AST work, only good class design, and is the safe, idiomatic way to offer an expressive mini-language.

In [ ]:
class Query:
    """A tiny fluent query builder, an internal DSL for describing a search."""
    def __init__(self):
        self._table = None
        self._conditions = []
        self._order = None

    def select_from(self, table):
        self._table = table
        return self                       # returning self enables chaining

    def where(self, condition):
        self._conditions.append(condition)
        return self

    def order_by(self, field):
        self._order = field
        return self

    def build(self):
        parts = [f"SELECT * FROM {self._table}"]
        if self._conditions:
            parts.append("WHERE " + " AND ".join(self._conditions))
        if self._order:
            parts.append(f"ORDER BY {self._order}")
        return " ".join(parts)

# The chained calls read almost like a sentence.
query = (Query()
         .select_from("users")
         .where("age > 18")
         .where("active = true")
         .order_by("name")
         .build())
print(query)

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Safe literal parsing (Easy)

In [ ]:
import ast
# Turn configuration text into real Python values, safely.
raw = "[10, 20, 30]"
values = ast.literal_eval(raw)
print(values, "sum:", sum(values))

### Example 2 — A reusable compiled expression (Easy)

In [ ]:
formula = compile("base * (1 + rate)", "<formula>", "eval")
for base, rate in [(100, 0.1), (200, 0.05)]:
    print(eval(formula, {"base": base, "rate": rate}))

### Example 3 — Counting names with the AST (Medium)

In [ ]:
import ast

class Counter(ast.NodeVisitor):
    def __init__(self):
        self.count = 0
    def visit_Name(self, node):
        self.count += 1
        self.generic_visit(node)

source = "a = b + c; d = a * b"
counter = Counter()
counter.visit(ast.parse(source))
print("name occurrences:", counter.count)

### Example 4 — Generating functions with exec (controlled input) (Medium)

In [ ]:
# Generate simple arithmetic functions from a trusted template.
# The operation symbol is restricted to a known safe set.
def make_operation(symbol):
    allowed = {"+", "-", "*"}
    if symbol not in allowed:
        raise ValueError("unsupported operation")
    namespace = {}
    exec(f"def op(a, b): return a {symbol} b", namespace)
    return namespace["op"]

add = make_operation("+")
mul = make_operation("*")
print("add(3, 4):", add(3, 4))
print("mul(3, 4):", mul(3, 4))

### Example 5 — Transforming code: constant folding of additions (Difficult)

In [ ]:
import ast

class FoldConstantAdds(ast.NodeTransformer):
    """Replace 'a + b' with the result when both sides are numeric literals."""
    def visit_BinOp(self, node):
        self.generic_visit(node)                       # transform children first
        if (isinstance(node.op, ast.Add)
                and isinstance(node.left, ast.Constant)
                and isinstance(node.right, ast.Constant)):
            folded = node.left.value + node.right.value
            return ast.copy_location(ast.Constant(value=folded), node)
        return node

source = "x = 2 + 3 + y"
tree = ast.parse(source, mode="exec")
FoldConstantAdds().visit(tree)
ast.fix_missing_locations(tree)

ns = {"y": 10}
exec(compile(tree, "<folded>", "exec"), ns)
print("result with constants folded:", ns["x"])      # (2+3) folded to 5, then + 10 = 15

### Example 6 — A fluent DSL for building text reports (Difficult)

In [ ]:
class Report:
    """A fluent builder for a small text report."""
    def __init__(self, title):
        self._title = title
        self._lines = []

    def heading(self, text):
        self._lines.append(text.upper())
        return self

    def item(self, text):
        self._lines.append(f"  - {text}")
        return self

    def spacer(self):
        self._lines.append("")
        return self

    def render(self):
        body = "\n".join(self._lines)
        bar = "=" * len(self._title)
        return f"{self._title}\n{bar}\n{body}"

document = (Report("Weekly Summary")
            .heading("Completed")
            .item("Shipped release 2.1")
            .item("Fixed login bug")
            .spacer()
            .heading("Planned")
            .item("Begin Track 5")
            .render())
print(document)

---

## Exercises

**Exercise 1 (Easy).** Use `ast.literal_eval` to safely turn the string `"{'x': 1, 'y': 2}"` into a dictionary, then print the value of `x`.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Use `eval` with a restricted namespace to evaluate the expression `"a + b"` where `a = 10` and `b = 5`, allowing only those two names.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Using `ast`, write a function `count_function_defs(source)` that returns how many function definitions (`def`) appear in a snippet of code. Test it on a snippet with two functions.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Write a fluent class `Pipeline` with methods `add(step)` (each returning `self`) and `run(value)` that applies every stored step (a function) to the value in order. Chain three steps and run a value through.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Using `ast.NodeVisitor`, write a function `uses_eval(source)` that returns `True` if the given code calls `eval` or `exec` anywhere, and `False` otherwise. Test it on a safe and an unsafe snippet.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import ast
data = ast.literal_eval("{'x': 1, 'y': 2}")
print(data["x"])

**Solution 2**

In [ ]:
print(eval("a + b", {"a": 10, "b": 5}))

**Solution 3**

In [ ]:
import ast

def count_function_defs(source):
    tree = ast.parse(source)
    return sum(1 for node in ast.walk(tree) if isinstance(node, ast.FunctionDef))

snippet = """
def one(): pass
def two(x): return x
value = 5
"""
print(count_function_defs(snippet))   # 2

**Solution 4**

In [ ]:
class Pipeline:
    def __init__(self):
        self.steps = []
    def add(self, step):
        self.steps.append(step)
        return self
    def run(self, value):
        for step in self.steps:
            value = step(value)
        return value

result = (Pipeline()
          .add(lambda x: x + 1)
          .add(lambda x: x * 2)
          .add(lambda x: x - 3)
          .run(5))
print(result)    # ((5+1)*2)-3 = 9

**Solution 5**

In [ ]:
import ast

def uses_eval(source):
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
            if node.func.id in {"eval", "exec"}:
                return True
    return False

print(uses_eval("x = sum([1, 2, 3])"))          # False
print(uses_eval("eval('2 + 2')"))               # True

---

## Summary and Key Points

- `eval` runs an expression and returns its value, `exec` runs statements and returns `None`, and `compile` produces a reusable code object; supplying namespaces limits available names.
- Never run untrusted input through `eval`/`exec`; restricting globals is not a security boundary. Use `ast.literal_eval` to parse literal values safely.
- Python parses source into an abstract syntax tree; `ast.parse` builds it and `ast.dump` shows its structure, enabling analysis without execution.
- `ast.NodeVisitor` walks the tree to analyse code (the basis of linters); `ast.NodeTransformer` rewrites it, after which `fix_missing_locations` and `compile` produce runnable code.
- An internal DSL with a fluent interface (methods returning `self`) offers an expressive, readable mini-language using ordinary class design, with none of the risks of `eval`.

### Track 4 complete

This concludes Track 4, Functional Programming & Metaprogramming. You can now treat functions as values and build closures, write decorators in all their forms, wield the functional toolkit, customise class creation with metaclasses and lighter hooks, introspect objects at runtime, and generate or analyse code safely. Track 5 builds on this with concurrency and asynchronous programming.